## LS

In [8]:
from wildetect.core.visualization.labelstudio_manager import LabelStudioManager
from wildata.converters.labelstudio.labelstudio_schemas import Result, Task
from pathlib import Path
from wildetect.core.data import DroneImage
from wildetect.core.config_models import FlightSpecs


In [2]:
ls_manager = LabelStudioManager(
    url="http://127.0.0.1:8080",
    api_key='4f3c25bad9334596c5b2c3b270a2d3105c8b5d4a',
    download_resources=False,
)

In [3]:
task = ls_manager.get_task(190628,as_sdk_task=False)

In [4]:
task

Task(id=190628, data=TaskData(image='/data/local-files?d=PhD\\Data per camp\\Dry season\\Kapiri\\Farm\\DJI_202310040946_014_Kapiri-Farm3\\DJI_20231004131109_0087.JPG'), annotations=[Annotation(id=69752, completed_by=1, result=[Result(id='688c565aebd3949892d2763d', type=<ResultType.RECTANGLELABELS: 'rectanglelabels'>, from_name='detections', to_name='image', original_width=8192, original_height=5460, image_rotation=0, value=ResultValue(x=34.619887932205295, y=0.1282051282051283, width=0.963608161544699, height=0.8651838062289492, rotation=0.0, rectanglelabels=['sable']), origin=<ResultOrigin.PREDICTION_CHANGED: 'prediction-changed'>, score=None), Result(id='688c565aebd3949892d2763e', type=<ResultType.RECTANGLELABELS: 'rectanglelabels'>, from_name='detections', to_name='image', original_width=8192, original_height=5460, image_rotation=0, value=ResultValue(x=37.7197265625, y=44.761904761904766, width=0.87890625, height=0.9523809523809524, rotation=0.0, rectanglelabels=['sable']), origin=<

In [5]:
task.image_path

'D:\\\\PhD\\Data per camp\\Dry season\\Kapiri\\Farm\\DJI_202310040946_014_Kapiri-Farm3\\DJI_20231004131109_0087.JPG'

In [6]:
Path(task.image_path).exists()

True

In [10]:
    flight_specs = FlightSpecs(sensor_height=24, focal_length=35, flight_height=180)

drone_image = DroneImage.from_ls_task(task=task,flight_specs=flight_specs)

In [12]:
drone_image.get_non_empty_annotations()

[Detection(bbox=[2836, 7, 2914, 54], confidence=1.0, class_id=0, class_name='sable', metadata=None, id='688c565aebd3949892d2763d', geographic_footprint=GeographicBounds(north=251008.55133229747, south=250823.33021229747, east=7349264.071694738, west=7349140.621094738, lat_center=-23.94951536111111, lon_center=30.552519166666666, width_px=8192, height_px=5460, gsd=2.261), gps_loc='23 56m 56.2565s S, 30 33m 8.13059s E, 0.831353km', image_gps_loc='23 56m 58.2553s S, 30 33m 9.069s E, 0.8313529999999999km', parent_image='D:\\\\PhD\\Data per camp\\Dry season\\Kapiri\\Farm\\DJI_202310040946_014_Kapiri-Farm3\\DJI_20231004131109_0087.JPG'),
 Detection(bbox=[3090, 2444, 3162, 2496], confidence=1.0, class_id=0, class_name='sable', metadata=None, id='688c565aebd3949892d2763e', geographic_footprint=GeographicBounds(north=251008.55133229747, south=250823.33021229747, east=7349264.071694738, west=7349140.621094738, lat_center=-23.94951536111111, lon_center=30.552519166666666, width_px=8192, height_px

In [13]:
drone_image.get_non_empty_predictions()

[]

## Load duplicates

In [7]:
import pandas as pd
import numpy as np

In [14]:
df_duplicated = pd.read_csv("../animal-duplicates.csv").rename(columns={"image": "task_id","bounding box":"bbox_id"})
df_duplicated.head(3)

,season,camp,Project ID,task_id,bbox_id,species,duplicate
0,Wet,K4_5,12,31346,dRlyzncAq0,buffalo,2
1,Wet,K4_5,12,31348,g1997mnNM0,buffalo,2
2,Wet,K4_5,12,31788,xOn9_xr2XK,roan,3


In [12]:
df_duplicated['bbox_id'].nunique()

245

In [13]:
df_loaded = pd.read_csv("../animal-duplicates_annotations.csv")
df_loaded.head(3)

,task_id,x_pixel,y_pixel,width_pixel,height_pixel,label,score,origin,bbox_id
0,39409,657.764755,34.688291,45.727425,34.074927,sable,0.492378,prediction,hbUXekrzK3
1,31346,735.785034,1128.890381,93.325897,60.050720,buffalo,0.823920,prediction,dRlyzncAq0
2,31788,1605.102190,553.608555,80.144758,52.590664,roan,0.586624,prediction,xOn9_xr2XK


In [11]:
df_loaded['bbox_id'].nunique()

98

In [15]:
df_loaded_pred = pd.read_csv("../animal-duplicates_predictions.csv")
df_loaded_pred.head(3)

,task_id,x_pixel,y_pixel,width_pixel,height_pixel,label,score,origin,bbox_id
0,39409,657.764755,34.688291,45.727425,34.074927,sable,0.492378,prediction,hbUXekrzK3
1,39411,671.625054,1101.370863,54.369485,26.958404,sable,0.347088,prediction,X22tB0VjlM
2,31346,735.785034,1128.890381,93.325897,60.050720,buffalo,0.823920,prediction,dRlyzncAq0


In [16]:
df_loaded_pred['bbox_id'].nunique()

240

In [17]:
intersection = np.intersect1d(df_loaded['bbox_id'], df_duplicated['bbox_id'])
intersection.shape

(98,)

In [18]:
intersection = np.intersect1d(df_loaded_pred['bbox_id'], df_duplicated['bbox_id'])
intersection.shape

(240,)